# Experiment 7 — Rollout-augmented Lyapunov analysis

This notebook is meant to replace the one-shot `run_exp7.py` workflow with an interactive workflow:

- load the trained `fhat` from Exp5,
- optionally retrain only the Lyapunov network `V` on random + rollout states,
- evaluate `stable`, `fhat`, and optionally baseline trajectories,
- plot rollout error, Lyapunov behaviour, projection fire rate, correction magnitude, and state trajectories.

Default mode is **analysis-only**: it loads `checkpoints/p4_rollout_aug_stable.pt` if it already exists.
Set `RUN_TRAIN = True` to regenerate Exp7 from inside the notebook.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision("high")

def find_repo_root(start: Path | None = None) -> Path:
    """Find repository root by walking upward until stable_icnn_physics/ is found."""
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "stable_icnn_physics").exists():
            return p
    raise RuntimeError("Could not find repo root. Start Jupyter from the DeepLearningFTN repo.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model, make_system
from stable_icnn_physics.data import (
    dataset_base_name,
    generate_derivative_data,
    load_dataset,
    save_dataset,
    tensor_dataset,
)
from stable_icnn_physics.eval import (
    autoregressive_rollout_model,
    lyapunov_decrease_values,
    rollout_error,
    rollout_system,
)
from stable_icnn_physics.train import evaluate_derivative_mse

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR   = REPO_ROOT / "data" / "cache"
CKPT_DIR    = REPO_ROOT / "checkpoints"
RESULTS_DIR = REPO_ROOT / "results"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("device:", DEVICE, "| torch:", torch.__version__)

## Configuration

`RUN_TRAIN=False` loads the existing Exp7 checkpoint and only evaluates/plots.  
Set `RUN_TRAIN=True` if you want to retrain `V` from the Exp5 `fhat` checkpoint.

In [ ]:
SEED = 0
TOLERANCE = 1e-5

P4_SYSTEM = "damped_pendulum_4"
P4_KWARGS = {"friction": 0.3, "gravity": 9.81}
P4_DT     = 0.02
P4_STEPS  = 300

# Exp7 training settings
RUN_TRAIN = False          # change to True to retrain V inside this notebook
EPOCHS_V  = 400
BATCH_V   = 512

N_TRAJ_TRAIN = 500
N_TRAJ_VAL   = 100

# Evaluation settings
N_EVAL_TRAJ = 16           # increase to 128/512 for stronger validation
EVAL_SEED   = SEED + 123

EXP5_STABLE_CKPT   = CKPT_DIR / "p4_random_large_e500_alpha1e5_stable.pt"
EXP5_BASELINE_CKPT = CKPT_DIR / "p4_random_large_e500_alpha1e5_baseline.pt"
EXP7_CKPT          = CKPT_DIR / "p4_rollout_aug_stable.pt"
EXP7_SUMMARY       = RESULTS_DIR / "p4_rollout_aug_notebook_summary.json"

print("RUN_TRAIN:", RUN_TRAIN)
print("Exp5 stable checkpoint:", EXP5_STABLE_CKPT.exists(), EXP5_STABLE_CKPT)
print("Exp7 checkpoint:", EXP7_CKPT.exists(), EXP7_CKPT)

## Load/generate data and system

The system is the 4-link damped pendulum, so the state dimension is 8:
`[theta_0, theta_1, theta_2, theta_3, omega_0, omega_1, omega_2, omega_3]`.

In [ ]:
p4r = make_system(P4_SYSTEM, **P4_KWARGS)
state_dim = p4r.state_dim

for split, n in [("train", 50_000), ("test", 10_000)]:
    path = CACHE_DIR / dataset_base_name(
        p4r, split=split, n_samples=n, seed=SEED, dataset_type="derivative"
    )
    if not path.exists():
        print(f"generating {split} data -> {path.name}")
        x, y = generate_derivative_data(p4r, n_samples=n, split=split, seed=SEED)
        save_dataset(path, x, y)
    else:
        print(f"reusing {split}: {path.name}")

p4r_train_path = CACHE_DIR / dataset_base_name(
    p4r, split="train", n_samples=50_000, seed=SEED, dataset_type="derivative"
)
p4r_test_path = CACHE_DIR / dataset_base_name(
    p4r, split="test", n_samples=10_000, seed=SEED, dataset_type="derivative"
)

x_p4r_train, y_p4r_train = load_dataset(p4r_train_path)
x_p4r_test,  y_p4r_test  = load_dataset(p4r_test_path)

p4r_train_ds = tensor_dataset(x_p4r_train, y_p4r_train)
p4r_test_ds  = tensor_dataset(x_p4r_test, y_p4r_test)

x0_eval = p4r.sample_initial_conditions(N_EVAL_TRAJ, split="test", seed=EVAL_SEED)
true_p4 = rollout_system(p4r, x0_eval, steps=P4_STEPS, dt=P4_DT)

print("system:", p4r.name, "| state_dim:", state_dim)
print("train:", x_p4r_train.shape, y_p4r_train.shape)
print("test:", x_p4r_test.shape, y_p4r_test.shape)
print("eval initial conditions:", x0_eval.shape)

## Build model and load Exp5 `fhat`

Exp7 reuses the nominal dynamics `fhat` learned in Exp5 and retrains only the Lyapunov network `V`.

In [ ]:
def make_stable():
    return build_stable_model(
        dim=state_dim,
        hidden=200,
        depth=3,
        lyapunov_hidden=100,
        lyapunov_eps=0.01,
        alpha=1e-5,
        rehu_width=0.01,
    )

if not EXP5_STABLE_CKPT.exists():
    raise FileNotFoundError(f"Missing Exp5 checkpoint: {EXP5_STABLE_CKPT}. Run run_exp5.py first.")

stable = make_stable()
stable.load_state_dict(torch.load(EXP5_STABLE_CKPT, map_location=DEVICE, weights_only=True)["model_state"])
stable.to(DEVICE).eval()

print("loaded Exp5 stable checkpoint")
print("fhat parameter count:", sum(p.numel() for p in stable.fhat.parameters()))
print("V parameter count:", sum(p.numel() for p in stable.V.parameters()))

## Optional: collect rollout states from `fhat`

This is the key Exp7 idea: train `V` not only on random states, but also on states that the nominal model actually visits during autoregressive rollout.

In [ ]:
def collect_rollout_states(fhat, system, n_trajs: int, steps: int, dt: float, split: str, seed: int) -> np.ndarray:
    """Roll out fhat from sampled initial conditions and collect every visited state."""
    ics = system.sample_initial_conditions(n_trajs, split=split, seed=seed)
    all_states = []

    fhat.eval()
    with torch.no_grad():
        for i in range(n_trajs):
            x = ics[i].copy()
            for _ in range(steps):
                all_states.append(x.copy())
                x_t = torch.tensor(x, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                xdot = fhat(x_t).squeeze(0).cpu().numpy()
                x = system.wrap_state(x + dt * xdot)

    return np.asarray(all_states, dtype=np.float32)

if RUN_TRAIN:
    rollout_train_states = collect_rollout_states(
        stable.fhat, p4r, N_TRAJ_TRAIN, P4_STEPS, P4_DT, "train", SEED + 1
    )
    rollout_val_states = collect_rollout_states(
        stable.fhat, p4r, N_TRAJ_VAL, P4_STEPS, P4_DT, "test", SEED + 2
    )

    combined_train = np.concatenate([x_p4r_train, rollout_train_states], axis=0)
    combined_val   = np.concatenate([x_p4r_test,  rollout_val_states], axis=0)

    print("rollout states train:", rollout_train_states.shape)
    print("rollout states val:  ", rollout_val_states.shape)
    print("combined train:", combined_train.shape)
    print("combined val:  ", combined_val.shape)
else:
    print("Skipping rollout-state collection because RUN_TRAIN=False.")

## Optional: train only `V`

The loss is:

\[
L_V = \mathbb{E}_x \left[\max(0, \nabla V(x)^\top \hat f(x) + \alpha V(x))^2 \right]
\]

`fhat` is frozen. Only `V` is updated.

In [ ]:
def train_V_rollout_augmented(stable, combined_train, combined_val):
    for p in stable.fhat.parameters():
        p.requires_grad_(False)
    stable.fhat.eval()

    optimizer = torch.optim.Adam(stable.V.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS_V, eta_min=1e-4
    )

    pin = DEVICE == "cuda"
    loader_train = DataLoader(
        TensorDataset(torch.from_numpy(combined_train)),
        batch_size=BATCH_V,
        shuffle=True,
        pin_memory=pin,
    )
    loader_val = DataLoader(
        TensorDataset(torch.from_numpy(combined_val)),
        batch_size=1024,
        shuffle=False,
        pin_memory=pin,
    )

    best_val_loss = float("inf")
    best_state = None
    history = {"epoch": [], "train_loss": [], "val_loss": [], "best_val_loss": []}

    for epoch in range(1, EPOCHS_V + 1):
        stable.V.train()
        total, count = 0.0, 0

        for (x_batch,) in loader_train:
            x_batch = x_batch.to(DEVICE, non_blocking=pin).requires_grad_(True)
            optimizer.zero_grad(set_to_none=True)

            with torch.enable_grad():
                fx = stable.fhat(x_batch)
                vx = stable.V(x_batch)
                gv = torch.autograd.grad(vx.sum(), x_batch, create_graph=True)[0]
                vio = (gv * fx).sum(dim=1, keepdim=True) + stable.alpha * vx
                loss = F.relu(vio).pow(2).mean()

            loss.backward()
            optimizer.step()

            total += loss.detach().item() * x_batch.shape[0]
            count += x_batch.shape[0]

        scheduler.step()
        train_loss = total / max(count, 1)

        if epoch % 20 == 0 or epoch == 1 or epoch == EPOCHS_V:
            stable.V.eval()
            val_total, val_count = 0.0, 0

            for (x_val,) in loader_val:
                x_val = x_val.to(DEVICE, non_blocking=pin).requires_grad_(True)
                with torch.enable_grad():
                    fx = stable.fhat(x_val)
                    vx = stable.V(x_val)
                    gv = torch.autograd.grad(vx.sum(), x_val, create_graph=False)[0]
                    vio = (gv * fx).sum(dim=1, keepdim=True) + stable.alpha * vx
                    val_loss_batch = F.relu(vio).pow(2).mean()

                val_total += val_loss_batch.item() * x_val.shape[0]
                val_count += x_val.shape[0]

            val_loss = val_total / max(val_count, 1)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.detach().clone() for k, v in stable.V.state_dict().items()}

            history["epoch"].append(epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["best_val_loss"].append(best_val_loss)

            print(
                f"epoch={epoch:04d} "
                f"train={train_loss:.6g} "
                f"val={val_loss:.6g} "
                f"best_val={best_val_loss:.6g}"
            )

    if best_state is not None:
        stable.V.load_state_dict(best_state)

    for p in stable.fhat.parameters():
        p.requires_grad_(True)

    stable.eval()
    return history

if RUN_TRAIN:
    history = train_V_rollout_augmented(stable, combined_train, combined_val)
    torch.save({"model_state": stable.state_dict(), "history": history}, EXP7_CKPT)
    print("saved:", EXP7_CKPT)
else:
    history = None
    if not EXP7_CKPT.exists():
        raise FileNotFoundError(
            f"RUN_TRAIN=False but Exp7 checkpoint is missing: {EXP7_CKPT}. "
            "Set RUN_TRAIN=True and rerun the training cells."
        )
    stable.load_state_dict(torch.load(EXP7_CKPT, map_location=DEVICE, weights_only=True)["model_state"])
    stable.to(DEVICE).eval()
    print("loaded existing Exp7 checkpoint:", EXP7_CKPT)

## Training curves

In [ ]:
if history is not None:
    plt.figure(figsize=(8, 5))
    plt.semilogy(history["epoch"], history["train_loss"], label="train")
    plt.semilogy(history["epoch"], history["val_loss"], label="val")
    plt.semilogy(history["epoch"], history["best_val_loss"], label="best val", linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("Lyapunov violation loss")
    plt.title("Exp7 V-only training loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No training history in analysis-only mode.")

## Evaluation helpers

Besides final/mean rollout error, this section computes:

- nominal Lyapunov violation: \(\nabla V(x)^T \hat f(x) + \alpha V(x)\)
- projection fire mask,
- correction magnitude \(\|f(x)-\hat f(x)\|\),
- discrete Lyapunov change \(V(x_{t+1}) - V(x_t)\).

In [ ]:
@torch.no_grad()
def load_baseline_if_available():
    if not EXP5_BASELINE_CKPT.exists():
        return None

    baseline = BaselineDynamicsMLP(dim=state_dim, hidden=200, depth=3)
    baseline.load_state_dict(
        torch.load(EXP5_BASELINE_CKPT, map_location=DEVICE, weights_only=True)["model_state"]
    )
    baseline.to(DEVICE).eval()
    return baseline

def compute_projection_diagnostics(model, traj: np.ndarray):
    """
    traj shape: (steps + 1, n_traj, dim)
    Returns arrays with shape (steps, n_traj), plus V with shape (steps + 1, n_traj).
    """
    model.eval()
    fire = []
    violation = []
    correction_norm = []
    v_values = []

    with torch.enable_grad():
        for t in range(traj.shape[0] - 1):
            x = torch.tensor(traj[t], dtype=torch.float32, device=DEVICE).requires_grad_(True)

            fx_hat = model.fhat(x)
            vx = model.V(x)
            gv = torch.autograd.grad(vx.sum(), x, create_graph=False)[0]

            vio = (gv * fx_hat).sum(dim=1) + model.alpha * vx.squeeze(1)
            denom = gv.square().sum(dim=1).clamp_min(model.denom_eps)
            corr = gv * (F.relu(vio).unsqueeze(1) / denom.unsqueeze(1))
            f_proj = fx_hat - corr

            fire.append((vio > 0).detach().cpu().numpy())
            violation.append(vio.detach().cpu().numpy())
            correction_norm.append(torch.linalg.norm(f_proj - fx_hat, dim=1).detach().cpu().numpy())
            v_values.append(vx.squeeze(1).detach().cpu().numpy())

    with torch.enable_grad():
        x_final = torch.tensor(traj[-1], dtype=torch.float32, device=DEVICE).requires_grad_(True)
        v_final = model.V(x_final).squeeze(1).detach().cpu().numpy()

    v_values.append(v_final)

    return {
        "fire": np.asarray(fire),
        "violation": np.asarray(violation),
        "correction_norm": np.asarray(correction_norm),
        "V": np.asarray(v_values),
    }

def discrete_delta_v(v_values: np.ndarray):
    return v_values[1:] - v_values[:-1]

def summarize_rollout(name: str, err: np.ndarray):
    return {
        f"{name}_final": float(err[-1]),
        f"{name}_mean": float(err.mean()),
        f"{name}_max": float(err.max()),
    }

## Run rollouts

In [ ]:
wrap = p4r.wrap_state

traj_stable = autoregressive_rollout_model(
    stable, x0_eval, steps=P4_STEPS, dt=P4_DT, device=DEVICE, wrap_fn=wrap
)
traj_fhat = autoregressive_rollout_model(
    stable.fhat, x0_eval, steps=P4_STEPS, dt=P4_DT, device=DEVICE, wrap_fn=wrap
)

baseline = load_baseline_if_available()
traj_baseline = None
if baseline is not None:
    traj_baseline = autoregressive_rollout_model(
        baseline, x0_eval, steps=P4_STEPS, dt=P4_DT, device=DEVICE, wrap_fn=wrap
    )

err_stable = rollout_error(p4r, true_p4, traj_stable).mean(axis=1)
err_fhat   = rollout_error(p4r, true_p4, traj_fhat).mean(axis=1)
err_baseline = None if traj_baseline is None else rollout_error(p4r, true_p4, traj_baseline).mean(axis=1)

diag = compute_projection_diagnostics(stable, traj_stable)
dV = discrete_delta_v(diag["V"])

decrease_random = lyapunov_decrease_values(stable, x_p4r_test[:2048], device=DEVICE).ravel()

summary = {
    "experiment": "p4_rollout_aug_notebook",
    "system": P4_SYSTEM,
    "alpha": stable.alpha,
    "n_eval_traj": N_EVAL_TRAJ,
    "rollout_steps": P4_STEPS,
    **summarize_rollout("stable_error", err_stable),
    **summarize_rollout("fhat_error", err_fhat),
    "lyapunov_max_violation_random_projected": float(decrease_random.max()),
    "lyapunov_fraction_satisfied_random_projected": float(np.mean(decrease_random <= TOLERANCE)),
    "projection_fire_rate": float(diag["fire"].mean()),
    "correction_norm_mean": float(diag["correction_norm"].mean()),
    "correction_norm_p95": float(np.quantile(diag["correction_norm"], 0.95)),
    "correction_norm_max": float(diag["correction_norm"].max()),
    "discrete_deltaV_fraction_nonpositive": float(np.mean(dV <= TOLERANCE)),
    "discrete_deltaV_max": float(dV.max()),
}

if err_baseline is not None:
    summary.update(summarize_rollout("baseline_error", err_baseline))

EXP7_SUMMARY.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("saved:", EXP7_SUMMARY)

## Plot: rollout error over time

In [ ]:
t = np.arange(P4_STEPS + 1) * P4_DT

plt.figure(figsize=(9, 5))
plt.plot(t, err_stable, label="stable / projected")
plt.plot(t, err_fhat, label="fhat only")
if err_baseline is not None:
    plt.plot(t, err_baseline, label="baseline")
plt.xlabel("time [s]")
plt.ylabel("mean state error")
plt.title("Rollout error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.semilogy(t, err_stable + 1e-12, label="stable / projected")
plt.semilogy(t, err_fhat + 1e-12, label="fhat only")
if err_baseline is not None:
    plt.semilogy(t, err_baseline + 1e-12, label="baseline")
plt.xlabel("time [s]")
plt.ylabel("mean state error, log scale")
plt.title("Rollout error, log scale")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Plot: projection behaviour

This is more informative than merely reporting `fraction_satisfied=1.0`, because the projected model is designed to satisfy the condition by construction. The important question is how often and how strongly the projection modifies `fhat`.

In [ ]:
t_mid = np.arange(P4_STEPS) * P4_DT

fire_rate_t = diag["fire"].mean(axis=1)
corr_mean_t = diag["correction_norm"].mean(axis=1)
corr_p95_t = np.quantile(diag["correction_norm"], 0.95, axis=1)

plt.figure(figsize=(9, 5))
plt.plot(t_mid, fire_rate_t)
plt.xlabel("time [s]")
plt.ylabel("fraction of trajectories")
plt.title("Projection fire rate over rollout")
plt.ylim(-0.02, 1.02)
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(t_mid, corr_mean_t, label="mean")
plt.plot(t_mid, corr_p95_t, label="p95")
plt.xlabel("time [s]")
plt.ylabel("||f(x) - fhat(x)||")
plt.title("Projection correction magnitude")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(diag["violation"].ravel(), bins=80)
plt.axvline(0.0, linestyle="--")
plt.xlabel("nominal violation = gradV·fhat + alpha V")
plt.ylabel("count")
plt.title("Nominal Lyapunov violation distribution on stable rollout states")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(diag["correction_norm"].ravel(), bins=80)
plt.xlabel("||f(x) - fhat(x)||")
plt.ylabel("count")
plt.title("Projection correction magnitude distribution")
plt.grid(True, alpha=0.3)
plt.show()

## Plot: Lyapunov value and discrete Lyapunov change

In [ ]:
V_mean_t = diag["V"].mean(axis=1)
V_p95_t = np.quantile(diag["V"], 0.95, axis=1)

plt.figure(figsize=(9, 5))
plt.plot(t, V_mean_t, label="mean V")
plt.plot(t, V_p95_t, label="p95 V")
plt.xlabel("time [s]")
plt.ylabel("V(x)")
plt.title("Lyapunov function along projected rollout")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

dV_mean_t = dV.mean(axis=1)
dV_p95_t = np.quantile(dV, 0.95, axis=1)
dV_max_t = dV.max(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(t_mid, dV_mean_t, label="mean ΔV")
plt.plot(t_mid, dV_p95_t, label="p95 ΔV")
plt.plot(t_mid, dV_max_t, label="max ΔV")
plt.axhline(0.0, linestyle="--")
plt.xlabel("time [s]")
plt.ylabel("V(x[t+1]) - V(x[t])")
plt.title("Discrete Lyapunov change under RK4 rollout")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("fraction ΔV <= tolerance:", np.mean(dV <= TOLERANCE))
print("max ΔV:", dV.max())
print("mean ΔV:", dV.mean())

## Plot: example trajectories

For each state coordinate, compare true trajectory, projected stable model, and `fhat only` for one selected initial condition.

In [ ]:
traj_id = 0
coord_names = [f"theta_{i}" for i in range(4)] + [f"omega_{i}" for i in range(4)]

for dim_idx, coord_name in enumerate(coord_names):
    plt.figure(figsize=(9, 4))
    plt.plot(t, true_p4[:, traj_id, dim_idx], label="true")
    plt.plot(t, traj_stable[:, traj_id, dim_idx], label="stable / projected")
    plt.plot(t, traj_fhat[:, traj_id, dim_idx], label="fhat only")
    if traj_baseline is not None:
        plt.plot(t, traj_baseline[:, traj_id, dim_idx], label="baseline")
    plt.xlabel("time [s]")
    plt.ylabel(coord_name)
    plt.title(f"Trajectory coordinate: {coord_name} | traj_id={traj_id}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## Optional: evaluate multiple seeds

Use this when you want stronger numbers for the final report. It can take longer.

In [ ]:
def evaluate_seed(eval_seed: int, n_eval_traj: int = 64):
    x0 = p4r.sample_initial_conditions(n_eval_traj, split="test", seed=eval_seed)
    true = rollout_system(p4r, x0, steps=P4_STEPS, dt=P4_DT)

    stable_traj = autoregressive_rollout_model(
        stable, x0, steps=P4_STEPS, dt=P4_DT, device=DEVICE, wrap_fn=wrap
    )
    fhat_traj = autoregressive_rollout_model(
        stable.fhat, x0, steps=P4_STEPS, dt=P4_DT, device=DEVICE, wrap_fn=wrap
    )

    e_stable = rollout_error(p4r, true, stable_traj).mean(axis=1)
    e_fhat = rollout_error(p4r, true, fhat_traj).mean(axis=1)

    seed_diag = compute_projection_diagnostics(stable, stable_traj)
    seed_dV = discrete_delta_v(seed_diag["V"])

    return {
        "seed": eval_seed,
        "n_eval_traj": n_eval_traj,
        "stable_final": float(e_stable[-1]),
        "stable_mean": float(e_stable.mean()),
        "fhat_final": float(e_fhat[-1]),
        "fhat_mean": float(e_fhat.mean()),
        "fire_rate": float(seed_diag["fire"].mean()),
        "corr_mean": float(seed_diag["correction_norm"].mean()),
        "corr_p95": float(np.quantile(seed_diag["correction_norm"], 0.95)),
        "dV_frac_nonpositive": float(np.mean(seed_dV <= TOLERANCE)),
        "dV_max": float(seed_dV.max()),
    }

RUN_MULTI_SEED = False
SEEDS = [123, 124, 125, 126, 127]

if RUN_MULTI_SEED:
    multi = [evaluate_seed(s, n_eval_traj=64) for s in SEEDS]
    multi_path = RESULTS_DIR / "p4_rollout_aug_multiseed_summary.json"
    multi_path.write_text(json.dumps(multi, indent=2), encoding="utf-8")

    try:
        import pandas as pd
        df = pd.DataFrame(multi)
        display(df)
        print(df.describe(numeric_only=True))
    except Exception:
        print(json.dumps(multi, indent=2))

    print("saved:", multi_path)
else:
    print("Set RUN_MULTI_SEED=True to run stronger multi-seed evaluation.")

## Notes for final report

Use these points if the plots support them:

1. Exp7 fixes the exp5/exp6 failure mode by training `V` on model-induced rollout states.
2. `fraction_satisfied=1.0` is expected for the projected dynamics; the more meaningful diagnostics are projection fire rate and correction magnitude.
3. If correction magnitude is small and rollout error remains close to or better than `fhat`, then the Lyapunov projection is no longer destructive.
4. Discrete `ΔV` is useful because the theoretical condition is continuous-time, while evaluation is performed through numerical RK4 rollouts.